# Diverse File Processing

**Scenario:** You're building a document ingestion pipeline for an enterprise search product. Your users upload everything — scanned PDFs from legal, photos of handwritten notes, Word reports from finance, HTML exports from internal wikis. Some files are small, others are 50MB+ contracts. You need a single pipeline that handles all of it reliably.

**What we'll build:**
1. Process a multi-page PDF contract and extract structured markdown
2. OCR a scanned image (photo of a receipt or whiteboard)
3. Compare `default` vs `performance` mode on the same document
4. Compare output formats (markdown, JSON, HTML) and when to use each
5. Handle large files (>15MB) — automatic offloading and URL-based input
6. Async job processing with polling for long-running documents
7. Build a production batch processing pattern combining jobs + large files

**Services used:** Document Intelligence, Jobs

**Prerequisites:**
- `pip install latence`
- `LATENCE_API_KEY` environment variable set
- Sample files in `data/test_files/` (PDF, image, DOCX)

In [2]:
import sys
sys.path.insert(0, "..")
from _test_utils import setup_notebook, print_section, print_subsection, print_success, time_operation, load_test_file, list_test_files

client = setup_notebook(api_key="lat_DhGuRk4jwbpfJbaerqB5Cp")

# Check what test files are available
print(f"Available test files: {list_test_files()}")


############################################################
# Latence SDK - Notebook Setup
############################################################

✅ Client initialized (base_url: https://api.latence.ai)
Available test files: ['end-to-end.ipynb', 'sample.jpg', 'sample.pptx', 'sample.docx', 'sample.pdf', 'zollrecht.pdf', 'test.png', 'test.pptx', 'test.docx', 'test.pdf', 'README.md', '.gitkeep']


## 1. Process a PDF Document

The most common case: take a PDF and get clean markdown. This works for both native PDFs and scanned documents — the service handles OCR automatically.

In [ ]:
# TODO: Process a real PDF and display the extracted content


## 2. OCR a Scanned Image

Photos from phones, scanned receipts, whiteboard snapshots — anything that's an image with text. The service auto-detects orientation and applies dewarping if needed.

In [ ]:
# TODO: Process an image file (receipt, whiteboard photo, or scanned doc)

## 3. Default vs Performance Mode

The service has two modes:
- **default** — fast, page-by-page processing
- **performance** — adds multi-page refinement (table merging, title hierarchy reconstruction, page concatenation)

Let's run the same document through both and compare quality vs speed.

In [ ]:
# TODO: Process same document in both modes, compare output and timing

## 4. Output Format Comparison

Different downstream consumers need different formats:
- **markdown** — for LLMs, RAG pipelines, human reading
- **json** — for structured extraction, databases
- **html** — for web display, preserving layout
- **xlsx** — for spreadsheet data, tabular content

In [ ]:
# TODO: Process same document with each output format, show the differences

## 5. Large File Handling (>15MB)

The gateway handles large files transparently:

| File size | What happens |
|-----------|-------------|
| **≤ 15MB** | Sent inline as `file_base64` — fast, simple |
| **15MB–100MB** | Gateway auto-uploads to B2 cloud storage, replaces with a presigned URL — no code changes needed |
| **> 100MB** | Rejected (413 FILE_TOO_LARGE) |

When using `file_path`, the SDK base64-encodes the file and the gateway handles the rest. For very large files, you can also provide a `file_url` directly — the gateway fetches it server-side, bypassing upload entirely.

**Best practice for large files:** Use `file_url` when possible (avoids base64 encoding overhead) and always set `return_job=True` since processing takes longer.

In [ ]:
# Option A: Local file (SDK handles base64 encoding, gateway handles B2 upload if >15MB)
# result = client.document_intelligence.process(
#     file_path="large_report.pdf",
#     return_job=True,  # Recommended for large files
# )

# Option B: URL (gateway fetches server-side — no upload overhead)
# result = client.document_intelligence.process(
#     file_url="https://your-bucket.s3.amazonaws.com/large_report.pdf",
#     return_job=True,
# )

# TODO: Demonstrate with a real large file or URL

## 6. Async Jobs with Polling

For large documents or when you don't want to block, use `return_job=True`. The gateway queues the work and gives you a job ID to poll.

Three approaches:
1. **`client.jobs.wait()`** — blocks until done (simplest)
2. **Manual polling** — check status periodically (for progress reporting)
3. **Fire and forget** — submit many jobs, collect results later

In [ ]:
# Approach 1: Wait (blocks until complete)
# job = client.document_intelligence.process(file_path="report.pdf", return_job=True)
# print(f"Job submitted: {job.job_id}")
# result = client.jobs.wait(job.job_id, timeout=300)
# print(f"Status: {result.status}")
# print(result.output)

# TODO: Submit a real document as an async job and wait for it

In [ ]:
# Approach 2: Manual polling (for progress reporting)
# import time
#
# job = client.document_intelligence.process(file_path="report.pdf", return_job=True)
# print(f"Job submitted: {job.job_id}")
#
# while True:
#     status = client.jobs.get(job.job_id)
#     print(f"  Status: {status.status}")
#
#     if status.status in ("COMPLETED", "CACHED", "PULLED"):
#         print(f"Done! Output: {len(str(status.output))} chars")
#         break
#     elif status.status == "FAILED":
#         print(f"Failed: {status.error_message}")
#         break
#
#     time.sleep(2)

# TODO: Demonstrate manual polling with a real document

In [ ]:
# Approach 3: Retrieve output separately (for B2-cached results)
# If the output is large, it may be stored in B2. Use retrieve() to fetch it.
#
# output = client.jobs.retrieve(job.job_id)  # Auto-fetches from B2 if needed

# TODO: Show retrieve pattern

## 7. Production Batch Processing

Real pipelines combine everything: mixed file types, large files, async jobs. Here's a production pattern that submits all files as background jobs, polls for completion, and collects results.

In [ ]:
# Production pattern: batch process with async jobs
#
# import time
# from pathlib import Path
#
# files = ["contract.pdf", "receipt.jpg", "report.docx", "notes.html"]
# jobs = {}
#
# # 1. Submit all files as background jobs
# for f in files:
#     job = client.document_intelligence.process(
#         file_path=f,
#         mode="performance",
#         return_job=True,
#     )
#     jobs[job.job_id] = f
#     print(f"Submitted {f} -> {job.job_id}")
#
# # 2. Poll until all complete
# results = {}
# pending = set(jobs.keys())
#
# while pending:
#     for job_id in list(pending):
#         status = client.jobs.get(job_id)
#         if status.status in ("COMPLETED", "CACHED", "PULLED"):
#             results[jobs[job_id]] = client.jobs.retrieve(job_id)
#             pending.discard(job_id)
#             print(f"  Completed: {jobs[job_id]}")
#         elif status.status == "FAILED":
#             results[jobs[job_id]] = {"error": status.error_message}
#             pending.discard(job_id)
#             print(f"  Failed: {jobs[job_id]} - {status.error_message}")
#     if pending:
#         time.sleep(3)
#
# # 3. Summary
# print(f"\nProcessed {len(results)} files")
# for filename, result in results.items():
#     if "error" in result:
#         print(f"  {filename}: FAILED - {result['error']}")
#     else:
#         content = result.get("content", "")
#         print(f"  {filename}: {len(content)} chars extracted")

# TODO: Run with real test files

---
## Cleanup

In [ ]:
client.close()
print("Done.")